# Scrapers for MyDramaList

## Kailyn Lau

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

## Table of Contents
1. [Research Question](#sec1)
2. [Setup](#sec2)
3. [Scraping Type 1 — Aggregation page (`/shows/top`)](#sec3)
4. [Scraping Type 2 — Dedicated page (Details + Recommendations)](#sec4)
5. [Route A — combining both: build a joined dataset across many dramas](#sec5)
6. [Export to CSV](#sec6)
7. [Reflection / challenges / GenAI use](#sec7)

<h2 id=sec1>Research Question</h2>

My goal is to explore what information is available about highly ranked dramas on MyDramaList and how information from aggregation pages can be combined with
information from individual drama pages.

I started by scraping the **Top Shows** aggregation page to collect basic information and URLs for individual dramas. From there, I'm interested in the following question:

Do dramas ranked in MyDramaList's Top 100 show a recency bias, i.e. are recently aired dramas (2023–2026) disproportionately represented compared to older, longer-running classics, and does original network (Netflix vs. others) correlate with how recent a top-ranked drama is?

To do this, I've scraped the **Details** and **Recommendations** sections for each of the top shows. This demonstrates how web automation can be used to build a richer, joined dataset by connecting information across multiple pages (Route A).

<h2 id=sec2>Setup</h2>

Imports and constants. We're use `seleniumbase` (as before) since MyDramaList uses Cloudflare and a plain `requests` + BeautifulSoup approach is blocked. (I did indeed try it and it didn't work, so now I know!)


In [1]:
import json
import time

import pandas as pd
from seleniumbase import Driver

BASE_URL = "https://mydramalist.com"
TOP_SHOWS_URL = f"{BASE_URL}/shows/top"

# How many aggregation pages / drama detail pages to visit.
# Technically there are 5000 results (225 pages), but I didn't want to wait for it to run, so the current code uses 5 aggregation pages
# and 25 individual dramas.
PAGES_TO_SCRAPE = 5        # aggregation pages (20 dramas each)
DRAMAS_TO_VISIT = 25        # how many of those dramas to open individually
REQUEST_DELAY_SECONDS = 1.5


<h2 id=sec3> Scraping Type 1 — Aggregation page </h2>

Parses one "card" from the Top Shows grid into a dict.

In [2]:
def parse_aggregation_card(card):
    """Parse a single drama card from a MyDramaList aggregation page
    (e.g. /shows/top) into a flat dict.
    """
    rank = card.find_element(
        "css selector", ".ranking span"
    ).text.strip()

    title_element = card.find_element(
        "css selector", ".title a"
    )
    title = title_element.text.strip()

    url = title_element.get_attribute("href")
    if url.startswith("/"):
        url = BASE_URL + url

    meta = card.find_element(
        "css selector", ".text-muted"
    ).text.strip()

    drama_type, rest = meta.split(" - ", 1)
    year, episodes = rest.split(", ", 1)

    rating = card.find_element(
        "css selector", ".score"
    ).text.strip()

    paragraphs = card.find_elements("css selector", "p")
    synopsis = paragraphs[-1].text.strip()

    return {
        "rank": rank,
        "title": title,
        "url": url,
        "type": drama_type,
        "year": year,
        "episodes": episodes,
        "rating": rating,
        "synopsis": synopsis,
    }

In [3]:
def scrape_aggregation_pages(num_pages=PAGES_TO_SCRAPE):
    """Scrape `num_pages` pages of the Top Shows list (20 dramas/page)."""
    all_dramas = []

    with Driver(page_load_strategy="eager") as driver:
        for page in range(1, num_pages + 1):
            url = f"{TOP_SHOWS_URL}?page={page}"
            print(f"Scraping aggregation page {page}...")

            driver.open(url)
            driver.sleep(REQUEST_DELAY_SECONDS)

            cards = driver.find_elements("css selector", ".row-cell.content")
            print(f"  Found {len(cards)} dramas")

            for card in cards:
                try:
                    all_dramas.append(parse_aggregation_card(card))
                except Exception as e:
                    print("  Could not parse card:", e)

    print(f"\nTotal dramas scraped from aggregation pages: {len(all_dramas)}")
    return all_dramas

<h2 id=sec3>Scraping Type 2 — Dedicated page (Details + Recommendations)</h2>

I originally had an error where `parse_details` was dropping fields, but I think that's because the Details information renders twice on every drama page?

From my estimation, it looks like it renders:

- once inside `#show-detailsxx .show-detailsxss`, in a list with the class
  `hidden-md-up` (CSS hides this copy at medium-and-up screen widths), and
- once again in the right sidebar, inside a box with the class `hidden-sm-down`
  (CSS hides this copy at small-and-down widths).

So again, media responsiveness and whatnot.

I looked into it, and it looks like Selenium's `.text` returns an empty string for anything CSS is currently hiding, so I believe some of the fields were hidden when they weren't supposed to be. Instead, I've decided to check both locations, keep only elements that are actually visible (`.is_displayed()`), and merge the results.


In [4]:
def parse_details(driver):
    """Scrape the Details section of a drama page.

    MyDramaList duplicates this content across two responsive copies
    (one for mobile-width layouts, one for desktop-width layouts), so we
    check both locations and only keep whichever copy the browser actually
    renders as visible.
    """
    details = {}

    selectors = [
        "#show-detailsxx .show-detailsxss .list-item",       # main-column copy
        ".col-lg-4 .box.hidden-sm-down .box-body.light-b .list-item",  # sidebar copy
    ]

    for selector in selectors:
        for item in driver.find_elements("css selector", selector):
            if not item.is_displayed():
                continue
            try:
                label_element = item.find_element("css selector", "b.inline")
            except Exception:
                continue

            label = label_element.text.strip().rstrip(":")
            text = item.text.strip()
            value = text[len(label) + 1:].strip()

            if label and label not in details:
                details[label] = value

    return details


### Recommendations section

The Recommendations box is a simple grid of poster thumbnails, each wrapped in an `<a>` with a `title` (the recommended drama's name) and an `href` (its URL).


In [5]:
def parse_recommendations(driver):
    """Scrape the Recommendations section of a drama page.

    Returns a list of {"title": ..., "url": ...} dicts.
    """
    recommendations = []

    rec_links = driver.find_elements(
        "css selector", ".details-recommendations .rec-item a"
    )

    for link in rec_links:
        title = link.get_attribute("title")
        url = link.get_attribute("href")
        if url and url.startswith("/"):
            url = BASE_URL + url
        recommendations.append({"title": title, "url": url})

    return recommendations


<h2 id=sec5> Route A — joining aggregation + dedicated pages </h2>

Here, I scrape the Top Shows aggregation page to get a list of dramas + URLs, then visit each of those URLs individually to pull the Details and Recommendations for each one, merging everything into a single joined dataset.

`DRAMAS_TO_VISIT`, as explained above, caps how many detail pages we actually open, since visiting all 5000 is very slow.


In [6]:
def build_joined_dataset(aggregation_dramas, limit=DRAMAS_TO_VISIT):
    """For each drama from the aggregation scrape (up to `limit`), visit its
    dedicated page and merge in Details + Recommendations. Returns a list of
    fully joined drama dicts.
    """
    joined = []

    with Driver(page_load_strategy="eager") as driver:
        for drama in aggregation_dramas[:limit]:
            print(f"Visiting: {drama['title']} ({drama['url']})")

            driver.open(drama["url"])
            driver.sleep(REQUEST_DELAY_SECONDS)

            details = parse_details(driver)
            recommendations = parse_recommendations(driver)

            record = dict(drama)  # copy the aggregation-page fields
            record.update(details)
            # Store recs as a JSON string so the row stays flat for a CSV;
            # a list of dicts can't live in a single spreadsheet cell.
            record["recommendations"] = json.dumps(recommendations)

            joined.append(record)

    return joined


<h2 id=sec6>Run it</h2>

No explanation needed!

In [7]:
aggregation_dramas = scrape_aggregation_pages()
joined_dataset = build_joined_dataset(aggregation_dramas)

print(f"\nJoined dataset size: {len(joined_dataset)}")
joined_dataset[0]

Scraping aggregation page 1...
  Found 20 dramas
Scraping aggregation page 2...
  Found 20 dramas
Scraping aggregation page 3...
  Found 20 dramas
Scraping aggregation page 4...
  Found 20 dramas
Scraping aggregation page 5...
  Found 20 dramas

Total dramas scraped from aggregation pages: 100
Visiting: When Life Gives You Tangerines (https://mydramalist.com/735043-life)
Visiting: Twinkling Watermelon (https://mydramalist.com/739603-sparkling-watermelon)
Visiting: Move to Heaven (https://mydramalist.com/49231-move-to-heaven)
Visiting: Weak Hero Class 1 (https://mydramalist.com/702267-weak-hero)
Visiting: Alchemy of Souls (https://mydramalist.com/52939-can-this-person-be-translated)
Visiting: The Trauma Code: Heroes on Call (https://mydramalist.com/54697-golden-hour)
Visiting: Pursuit of Jade (https://mydramalist.com/760409-zhu-yu)
Visiting: Hospital Playlist Season 2 (https://mydramalist.com/57173-hospital-playlist-2)
Visiting: Flower of Evil (https://mydramalist.com/54625-flower-of-ev

{'rank': '#1',
 'title': 'When Life Gives You Tangerines',
 'url': 'https://mydramalist.com/735043-life',
 'type': 'Korean Drama',
 'year': '2025',
 'episodes': '16 episodes',
 'rating': '9.3',
 'synopsis': "It is a story that resembles a tribute to our parents' tender and still youthful seasons when they were so young, including the story of mother's first love, father's heroic tales, grandma's rebellious youth, and grandpa's…",
 'Native Title': '폭싹 속았수다',
 'Also Known As': 'Insaeng , Life , Pogssag Sogassuda , Pokssak Sogasssuda , Sugo Manheusyeossseubnida , Thank You For Your Hard Work , You Have Done Well , You Were Fooled , 수고 많으셨습니다 , 인생',
 'Director': 'Kim Won Suk',
 'Screenwriter': 'Im Sang Choon',
 'Genres': 'Romance, Life, Drama',
 'Tags': 'Family Relationship, Nice Male Lead, Poor Male Lead, Teenage Pregnancy, Father-Daughter Relationship, Heartwarming, Break Up, Hardworking Female Lead, Healthy Mains’ Relationship, Childhood Sweethearts (Vote tags)',
 'Title': 'When Life Gi

<h2 id=sec7>Export to CSV</h2>

Once `joined_dataset` is populated (from the cell above), convert it to a
DataFrame and write it out.


In [8]:
df = pd.DataFrame(joined_dataset)
df.to_csv("mydramalist_top_shows_joined.csv", index=False)
df.head()

,rank,title,url,type,year,episodes,rating,synopsis,Native Title,Also Known As,...,Original Network,Duration,Content Rating,Score,Ranked,Popularity,Watchers,recommendations,Related Content,Screenwriter & Director
0,#1,When Life Gives You Tangerines,https://mydramalist.com/735043-life,Korean Drama,2025,16 episodes,9.3,It is a story that resembles a tribute to our ...,폭싹 속았수다,"Insaeng , Life , Pogssag Sogassuda , Pokssak S...",...,Netflix,1 hr. 2 min.,13+ - Teens 13 or older,"9.3 (scored by 82,145 users)",#7,#38,"179,080","[{""title"": ""Our Blues"", ""url"": ""https://mydram...",NaN,NaN
1,#2,Twinkling Watermelon,https://mydramalist.com/739603-sparkling-water...,Korean Drama,2023,16 episodes,9.2,"In 2023, high school student Eun Gyeol, a CODA...",반짝이는 워터멜론,"Banjjagineun Woteomellon , Shining Watermelon ...",...,tvN,1 hr. 10 min.,15+ - Teens 15 or older,"9.2 (scored by 117,074 users)",#24,#15,"236,002","[{""title"": ""Lovely Runner"", ""url"": ""https://my...",NaN,NaN
2,#3,Move to Heaven,https://mydramalist.com/49231-move-to-heaven,Korean Drama,2021,10 episodes,9.1,Han Geu Ru is an autistic 20-year-old guy. He ...,무브 투 헤븐: 나는 유품정리사입니다,Move To Heaven: I Am a Person Who Arranges Art...,...,Netflix,52 min.,18+ Restricted (violence & profanity),"9.1 (scored by 79,720 users)",#34,#35,"184,147","[{""title"": ""Tomorrow"", ""url"": ""https://mydrama...",NaN,NaN
3,#4,Weak Hero Class 1,https://mydramalist.com/702267-weak-hero,Korean Drama,2022,8 episodes,9.1,Yeon Shi Eun is a model student who ranks at t...,약한영웅 Class 1,"Weak Hero , Weak Hero Season 1 , Yakhanyeongun...",...,Wavve,40 min.,18+ Restricted (violence & profanity),"9.1 (scored by 128,788 users)",#36,#11,"240,943","[{""title"": ""Study Group"", ""url"": ""https://mydr...",Weak Hero Class 2 (Korean sequel)\nWeak Hero C...,Yoo Su Min
4,#5,Alchemy of Souls,https://mydramalist.com/52939-can-this-person-...,Korean Drama,2022,20 episodes,9.1,"In the fictional country of Daeho, young mages...",환혼,"Alchemy of Souls Part 1 , Alchemy of Souls Sea...",...,tvN,1 hr. 20 min.,15+ - Teens 15 or older,"9.1 (scored by 118,345 users)",#37,#13,"239,131","[{""title"": ""Alchemy of Souls Season 2: Light a...",Alchemy of Souls Season 2: Light and Shadow (K...,NaN


In [9]:
df= pd.DataFrame(joined_dataset)

<h2 id=sec7>Reflection</h2>

**Research Question response:** All of this was by me eyeballing the results, although given more time, I would write a script to actually analyze and visualize the results.

Based on the top 25 samples of data, it looks like there's a mild recency skew but not an overwhelming one. Of the 25 top-ranked dramas, 36% (9/25) aired in 2023 or later, while 64% predate that, including an interesting 2005 drama (*One Liter of Tears*) and two from 2015! MyDramaList's Top ranking looks like it rewards sustained community rating rather than pure recency, though there's still a cluster of newer titles.

Original network shows a more interesting pattern than expected: **tvN dominates** with 10 of 25 titles (40%), far ahead of Netflix's 4. But the 4 Netflix titles skew notably newer — average release year 2024.2, versus 2020.0 for everything else, including two 2025/2026 titles. That's consistent with Netflix's original-content strategy in Korea ramping up more recently, while tvN's presence is built on a longer back-catalog of acclaimed dramas (*Reply 1988* from 2015, *Crash Landing on You* from 2019, etc.) that have simply had more time to accumulate high scores and large rating counts.

It is important to note that this only represents 25 of the 5000 top dramas, since I had limited time, and that "Original Network" sometimes lists multiple platforms (e.g., `iQiyi, Tencent Video`), so a stricter analysis would need to decide how to bucket co-productions rather than treating each string as its own category.

**What worked:** The aggregation page scraper worked on the first try -- the card structure (`.row-cell.content`) is consistent and easy to parse.

**What didn't work initially:** `parse_details` was returning incomplete results. Debugging with Claude, I found MyDramaList duplicates the Details list into two responsive copies (mobile vs. desktop widths), and whichever one CSS was hiding at the browser's current window size came back as empty text from Selenium. The fix was to check both locations and filter for `.is_displayed()`.

**Challenge / scope decision:** I originally attempted a comment scraper (Route B), but the "Load more comments" button interaction didn't work reliably, so I switched to Route A (joining aggregation + dedicated pages) instead, plus a Recommendations scraper as my second required section.

**GenAI use:** I used Claude to help debug why `parse_details` was dropping fields (it identified the duplicate responsive-layout issue by reading the page's HTML), as well as a couple of other bugs. I also googled some syntax problems, and the Gemini response was occasionally helpful.